# **Evaluación de la Robustez de Modelos de Detección Deepfake en Redes Móviles 3G y 4G**

<img src="https://www.udistrital.edu.co/themes/custom/versh/images/default/preloader.png" align="left" width="192px" height="192px"/>
<img align="left" width="0" height="192px" hspace="10"/>

> Carlos Stiven Mora-Hoyos
> Juan Felipe Rodríguez Galindo  - **COD. 20181020158**
<br></br>
[![License](https://img.shields.io/badge/License-GPL_V.3-blue?style=flat-square)](https://www.gnu.org/licenses/gpl-3.0.html)


Introducción a la experimentación

Desarrollo del pipeline propuesto para evaluar modelos (XceptionNet, EfficientNet-B4, ViT) bajo degradaciones de redes móviles (3G/4G) y transcodificación de video.

## **Desarrollo**

In [ ]:
# Librerias necesarias
import os
import subprocess
import glob
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import transforms
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

try:
    import timm
except ImportError:
    print("Instalando dependencias faltantes...")
    subprocess.run(["pip", "install", "timm", "pandas", "scikit-learn", "opencv-python"])
    import timm

try:
    subprocess.run(["ffmpeg", "-version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
except FileNotFoundError:
    raise RuntimeError("FFmpeg no se encuentra instalado en el sistema.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entorno listo. Utilizando aceleracion en: {device}")

In [ ]:
# Generador de Videos de Prueba
def create_dummy_dataset(base_path="./dataset_prueba"):
    os.makedirs(f"{base_path}/real", exist_ok=True)
    os.makedirs(f"{base_path}/fake", exist_ok=True)
    
    def generate_noise_video(filename, frames=30, width=640, height=480):
        if os.path.exists(filename): return
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(filename, fourcc, 30.0, (width, height))
        for _ in range(frames):
            frame = np.random.randint(0, 255, (height, width, 3), dtype=np.uint8)
            out.write(frame)
        out.release()
    
    generate_noise_video(f"{base_path}/real/real_01.mp4")
    generate_noise_video(f"{base_path}/fake/fake_01.mp4")
    print(f"Dataset de prueba inicializado en: {base_path}")
    return base_path

DATASET_PATH = create_dummy_dataset()

In [ ]:
# Lector de Dataset
def get_video_paths(data_dir):
    real_videos = glob.glob(os.path.join(data_dir, "real", "*.mp4"))
    fake_videos = glob.glob(os.path.join(data_dir, "fake", "*.mp4"))
    
    dataset = [(path, 0) for path in real_videos] + [(path, 1) for path in fake_videos]
    np.random.shuffle(dataset)
    return dataset

videos_dataset = get_video_paths(DATASET_PATH)
print(f"Total de videos cargados: {len(videos_dataset)}")

In [ ]:
# Modulo de Degradacion FFmpeg
def apply_degradation_pipeline(input_video, encoding_stage, network_stage):
    os.makedirs("./temp_videos", exist_ok=True)
    filename = os.path.basename(input_video)
    
    temp_enc_video = f"./temp_videos/enc_{encoding_stage}_{filename}"
    base_cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", input_video]
    
    if encoding_stage == "H.264":
        cmd_enc = base_cmd + ["-c:v", "libx264", "-crf", "23", temp_enc_video]
    elif encoding_stage == "H.265":
        cmd_enc = base_cmd + ["-c:v", "libx265", "-crf", "28", temp_enc_video]
    elif encoding_stage == "H.264-B":
        cmd_enc = base_cmd + ["-c:v", "libx264", "-b:v", "2M", "-maxrate", "2M", "-bufsize", "1M", temp_enc_video]
    elif encoding_stage == "H.265-B":
        cmd_enc = base_cmd + ["-c:v", "libx265", "-b:v", "1.5M", "-maxrate", "1.5M", "-bufsize", "1M", temp_enc_video]
    elif encoding_stage == "720P":
        cmd_enc = base_cmd + ["-vf", "scale=-2:720", "-c:v", "libx264", temp_enc_video]
    elif encoding_stage == "480P":
        cmd_enc = base_cmd + ["-vf", "scale=-2:480", "-c:v", "libx264", temp_enc_video]
    else:
        cmd_enc = base_cmd + ["-c", "copy", temp_enc_video]

    subprocess.run(cmd_enc)

    final_video = f"./temp_videos/final_{encoding_stage}_{network_stage}_{filename}"
    base_cmd_net = ["ffmpeg", "-y", "-loglevel", "error", "-i", temp_enc_video]
    
    if network_stage == "3G":
        cmd_net = base_cmd_net + ["-r", "15", "-b:v", "1M", "-c:v", "libx264", final_video]
    elif network_stage == "4G":
        cmd_net = base_cmd_net + ["-r", "30", "-b:v", "5M", "-c:v", "libx264", final_video]
    else:
        cmd_net = base_cmd_net + ["-c", "copy", final_video]

    subprocess.run(cmd_net)
    
    if os.path.exists(temp_enc_video):
        os.remove(temp_enc_video)
        
    return final_video

In [ ]:
# Carga de Modelos Reales usando Timm
def load_deepfake_models():
    print("Cargando Modelos CNN y ViT...")
    models_dict = {}
    
    model_xcp = timm.create_model('xception', pretrained=True, num_classes=2).to(device)
    model_xcp.eval()
    models_dict["XceptionNet"] = model_xcp
    
    model_eff = timm.create_model('efficientnet_b4', pretrained=True, num_classes=2).to(device)
    model_eff.eval()
    models_dict["EfficientNet-B4"] = model_eff
    
    model_vit = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=2).to(device)
    model_vit.eval()
    models_dict["Vision Transformer (ViT)"] = model_vit
    
    return models_dict

models_dict = load_deepfake_models()

preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
# Extraccion de Fotogramas y Prediccion
def predict_video_full(video_path, model, num_frames=10):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames <= 0: return 0.5
    
    frame_idxs = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    frames_tensor = []
    
    for idx in frame_idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            input_tensor = preprocess(frame_rgb)
            frames_tensor.append(input_tensor)
            
    cap.release()
    
    if not frames_tensor: return 0.5
    
    batch = torch.stack(frames_tensor).to(device)
    
    with torch.no_grad():
        outputs = model(batch)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        fake_prob = probs[:, 1].mean().item()
        
    return fake_prob

In [ ]:
# Bucle Central del Experimento
encoding_stages = ["Pristine", "H.264", "H.265", "H.264-B", "720P", "480P"]
network_stages = ["Ideal", "4G", "3G"]

experiment_results = []

print("\nINICIANDO PIPELINE DE EVALUACION")
for video_path, true_label in videos_dataset:
    print(f"\nProcesando Video: {os.path.basename(video_path)} (Etiqueta Real: {'Fake' if true_label==1 else 'Real'})")
    
    for enc in encoding_stages:
        for net in network_stages:
            
            if enc == "Pristine" and net == "Ideal":
                processed_video = video_path
                print("  -> Evaluando copia Pristina (Sin procesar)")
            elif enc == "Pristine" or net == "Ideal":
                continue
            else:
                processed_video = apply_degradation_pipeline(video_path, enc, net)
                print(f"  -> Pipeline aplicado: [{enc}] + [{net}]")
            
            for model_name, model in models_dict.items():
                prob_fake = predict_video_full(processed_video, model, num_frames=5)
                
                experiment_results.append({
                    "Video": os.path.basename(video_path),
                    "True_Label": true_label,
                    "Encoding": enc,
                    "Network": net,
                    "Model": model_name,
                    "Prob_Fake": prob_fake
                })
                
            if processed_video != video_path and os.path.exists(processed_video):
                os.remove(processed_video)

print("\nEvaluaciones finalizadas.")

In [ ]:
# Calculo de Metricas y Caida de Robustez
df = pd.DataFrame(experiment_results)
df['Pred_Class'] = (df['Prob_Fake'] >= 0.5).astype(int)

metrics_summary = []

for model in df['Model'].unique():
    for enc in encoding_stages:
        for net in network_stages:
            subset = df[(df['Model'] == model) & (df['Encoding'] == enc) & (df['Network'] == net)]
            
            if subset.empty: continue
            
            y_true = subset['True_Label'].tolist()
            y_prob = subset['Prob_Fake'].tolist()
            y_pred = subset['Pred_Class'].tolist()
            
            try:
                auc = roc_auc_score(y_true, y_prob)
            except ValueError:
                auc = np.nan
                
            acc = accuracy_score(y_true, y_pred)
            f1 = f1_score(y_true, y_pred, zero_division=0)
            
            metrics_summary.append({
                "Modelo": model,
                "Codificacion": enc,
                "Red": net,
                "AUC": auc,
                "Accuracy": acc,
                "F1-Score": f1
            })

df_metrics = pd.DataFrame(metrics_summary)

df_pristine = df_metrics[(df_metrics['Codificacion'] == 'Pristine')].copy()
df_pristine = df_pristine.set_index('Modelo')[['AUC']].rename(columns={'AUC': 'AUC_Base'})

df_final = df_metrics.merge(df_pristine, on='Modelo', how='left')
df_final['Caida_Robustez (Drop)'] = df_final['AUC_Base'] - df_final['AUC']

print("\nTABLA DE RESULTADOS - CAIDA DE ROBUSTEZ")
vista_resultados = df_final[df_final['Codificacion'] != 'Pristine'].sort_values(by=['Modelo', 'Red'])
print(vista_resultados.to_string(index=False, float_format="%.3f"))

vista_resultados.to_csv("resultados_robustez.csv", index=False)
print("\nResultados guardados exitosamente en 'resultados_robustez.csv'")

<hr>

                             Fin del Experimento - Evaluación de Robustez
> Carlos Stiven Mora-Hoyos & Juan Felipe Rodríguez Galindo